In [72]:
# env 파일 읽어오기

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

MODEL = 'gpt-6-luna'

In [73]:
from openai import OpenAI
import base64

client = OpenAI()

In [74]:
response = client.responses.create(
    model=MODEL,
    input="2026년 11월 11일 삼성전자 주식 가격을 알려줘",
)

print(response.output_text)

2026년 11월 11일은 아직 오지 않은 날짜라 삼성전자의 실제 주가는 알 수 없습니다. 해당 날짜가 지난 뒤 한국거래소나 증권사 앱에서 종가를 확인해 주세요. 원하시면 현재 정보를 바탕으로 주가 전망을 설명해 드릴 수 있지만, 전망은 확정 가격이 아닙니다.


#### Function Calling
- AI 모델이 사용자가 요청한 작업을 직접 처리할 수 없을 때 외부에서 미리 정의한 함수를 호출하도록 요청
- 1. 함수 정의
- 2. 함수 호출추 - AI가 함수호출을 요청
- 3. 함수 실행 - 개발자의 프로그램이 함수를 실행

In [75]:
response = client.responses.create(
    model=MODEL,
    input="10과 20을 더해줘",
)

print(response.output_text)

10 + 20 = 30입니다.


In [76]:
# 함수 정의

def add(a,b):
    return a+b

In [77]:
# 함수 의미 작성
tools = [
    {
        "type":"function",
        "name":"add",
        "description":"두 숫자를 더하는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                },
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    }
]

In [78]:
response = client.responses.create(
    model=MODEL,
    input="10과 20을 더해줘",
    tools=tools
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"a":10,"b":20}', call_id='call_1siRI9icx95RxX40y7b8zlNo', name='add', type='function_call', id='fc_0b91ecc9ff7056cb006af66c89836087d2b65bfefd54ce4bf5', async_=None, caller=None, namespace=None, status='completed')]


In [79]:
# 실제 함수 실행
import json


for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)    #문자열 '{"a":10,"b":20}'을 딕셔너리 {"a": 10, "b": 20}으로 바꿈
        # 함수 호출
        result = add(args['a'], args['b'])
        print(result)

30


In [80]:
response = client.responses.create(
    model=MODEL,
    previous_response_id=response.id,
    input=[
        {
            "type":"function_call_output",
            "call_id":item.call_id,
            "output":str(result)
        }
    ]
)

print(response.output_text)

10과 20의 합은 30입니다.


In [81]:
# 1. 첫 요청
response = client.responses.create(
    model=MODEL,
    input="10과 20을 더하고, 3과 4도 더해줘",
    tools=tools
)

# 2. 함수 요청마다 실행하고, 결과를 리스트에 모으기
outputs = []

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = add(args["a"], args["b"])

        outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,      # 이 요청의 번호표
            "output": str(result)
        })

# 3. 함수 요청이 있었을 때만 결과를 돌려주기
if outputs:
    response = client.responses.create(
        model=MODEL,
        previous_response_id=response.id,
        input=outputs,
        tools=tools
    )

print(response.output_text)

10과 20의 합은 30이고, 3과 4의 합은 7입니다.


### 사용자의 입력에 따라 다른 함수 호출
- 100에서 30을 빼고 20을 곱해줘

In [82]:
def add(a,b):
    return  a+b

def subtract(a,b):
    return  a-b

def multiply(a,b):
    return  a*b

def divide(a,b):
    return  a/b

In [83]:
# 함수 의미 작성
tools = [
    {
        "type":"function",
        "name":"add",
        "description":"두 숫자를 더하는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                },
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    },
    {
        "type":"function",
        "name":"subtract",
        "description":"두 숫자를 빼는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                },
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    },
        {
        "type":"function",
        "name":"multiply",
        "description":"두 숫자를 곱하는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                },
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    },
        {
        "type":"function",
        "name":"divide",
        "description":"두 숫자를 나누는 함수",
        "parameters":{
            "type":"object",
            "properties":{
                "a":{
                    "type":"number",
                    "description":"첫 번째 숫자"
                },
                "b":{
                    "type":"number",
                    "description":"두 번째 숫자"
                },
            },
            "required":["a","b"],
            "additionalProperties":False
        },
        "strict":True
    }
]

In [84]:
import json

response = client.responses.create(
    model=MODEL,
    input="100에서 30을 빼고 20을 곱해줘",
    tools=tools
)

print(response.output)

# function call이 있으면 함수 실행
function_registry={
    "add":add,
    "subtract":subtract,
    "multiply":multiply,
    "divide":divide
}

while True:
    function_calls = [item for item in response.output if item.type=="function_call"]

    if not function_calls:
        break

    tool_outputs=[]

    for item in function_calls:
        args=json.loads(item.arguments)      # '{"a":10, "b":20}' -> 딕셔너리로

        func = function_registry[item.name]    # AI가 요청한 함수 이름
        result = func(**args)

        tool_outputs.append(        
            {
            "type":"function_call_output",
            "call_id":item.call_id,
            "output":str(result)
            }
        )


    response = client.responses.create(
        model=MODEL,
        input=tool_outputs,
        previous_response_id=response.id,
        tools=tools
    )

print(response.output_text)


[ResponseFunctionToolCall(arguments='{"a":100,"b":30}', call_id='call_1KEicELF4CqxeG7TqjldmnQK', name='subtract', type='function_call', id='fc_04684ffb1f8c6d53006af66c91774087d2afb3458090d415d4', async_=None, caller=None, namespace=None, status='completed')]
1,400입니다.


In [87]:
documents = [
    {
        "title":"AI 기술 보고서",
        "content":"LLM과 LAG 시스템의 구조를 설명한다."
    },
    {
        "title":"보안 기술 보고서",
        "content":"PQC와 양자내성암호를 설명한다."
    },
    {
        "title":"추천 시스템 보고서",
        "content":"사용자 행동 데이터를 이용한 추천 모델이다."
    }
]

def search_documents(keyword):
    results = []

    for doc in documents:
        if keyword.lower() in (doc['title'] + doc['content']).lower():
            results.append(doc)

    return results

tools = [
    {
        "type":"function",
        "name":"search_documents",
        "description":"회사 내부 문서를 키워드로 검색한다.",
        "parameters":{
            "type":"object",
            "properties":{
                "keyword":{
                    "type":"string",
                    "description":"검색 키워드"
                },
            },
            "required":["keyword"],
            "additionalProperties":False
        },
        "strict":True
    },
]


response = client.responses.create(
    model=MODEL,
    input="AI 관련 사내 문서를 찾아줘",
    tools=tools,
)

print(response.output)

while True:
    function_calls = [item for item in response.output if item.type=="function_call"]

    if not function_calls:
        break

    tool_outputs=[]

    for item in function_calls:
        args=json.loads(item.arguments)      # '{"a":10, "b":20}' -> 딕셔너리로
        result = search_documents(**args)

        tool_outputs.append(        
            {
            "type":"function_call_output",
            "call_id":item.call_id,
            "output":json.dumps(result, ensure_ascii=False)     # dict 구조일때 변환
            }
        )

    response = client.responses.create(
        model=MODEL,
        input=tool_outputs,
        previous_response_id=response.id,
        tools=tools
    )

print(response.output_text)

[ResponseFunctionToolCall(arguments='{"keyword":"AI 인공지능 관련 사내 문서"}', call_id='call_Sj2ySjn9IvzWPbeqwZtvh6D1', name='search_documents', type='function_call', id='fc_021ae4960ab8cd5d006af66d28fc0887d292e943585973aa00', async_=None, caller=None, namespace=None, status='completed')]
‘AI 인공지능 관련 사내 문서’를 검색했지만 결과가 없습니다. ‘생성형 AI’, ‘ChatGPT’, ‘머신러닝’처럼 구체적인 키워드로 다시 찾아볼까요?


In [89]:
# uv add yfinance

import yfinance as yf

stock = yf.Ticker("005930.KS")
# data = stock.history(start="2026-11-09")
data=stock.history(period="1d")
print(data)

                               Open      High       Low     Close   Volume  \
Date                                                                         
2026-11-13 00:00:00+09:00  214000.0  216000.0  212000.0  213000.0  3716663   

                           Dividends  Stock Splits  
Date                                                
2026-11-13 00:00:00+09:00        0.0           0.0  


In [90]:
def get_korean_stock_price(code:str):
    symbol = f"{code}.KS"
    stock = yf.Ticker(symbol)
    data = stock.history(period = "1d")

    if data.empty:
        return json.dumps({"error" : "주식 데이터를 찾을 수 없습니다."})

    # 마지막 행 가져오기
    latest = data.iloc[-1]

    return json.dumps({
        "code":code,
        "current_price":float(latest['Close']),
        "open":float(latest['Open']),
        "High":float(latest['High']),
        "Low":float(latest['Low']),
        "Volume":float(latest['Volume']),
    })

tools = [
    {
        "type":"function",
        "name":"get_korean_stock_price",
        "description":"한국 주식 종목코드를 입력받아 가격을 조회합니다.",
        "parameters":{
            "type":"object",
            "properties":{
                "code":{
                    "type":"string",
                    "description":"6자리 한국 주식 종목 코드(예: 005930)"
                },
            },
            "required":["code"],
            "additionalProperties":False
        },
        "strict":True
    },
]


def run_korean_stock_agent(user_input:str):
    response = client.responses.create(
        model=MODEL,
        input=user_input,
        tools=tools
    )

    function_calls = [item for item in response.output if item.type == "function_call"]

    if not function_calls:
        return response.output_text

    tool_outputs=[]

    for item in function_calls:
        args=json.loads(item.arguments)       
        result = get_korean_stock_price(**args)

        tool_outputs.append(        
            {
            "type":"function_call_output",
            "call_id":item.call_id,
            "output":result
            }
        )


    final_response = client.responses.create(
        model=MODEL,
        input=tool_outputs,
        previous_response_id=response.id,
        tools=tools
    )

    return final_response.output_text


run_korean_stock_agent("005380")

'현대자동차(005380) 현재가 **278,500원**입니다.  \n시가 269,000원, 고가 279,000원, 저가 268,000원, 거래량 255,963주입니다.'

In [93]:
# openai 

response = client.responses.create(
    model=MODEL,
    reasoning={"effort":"low"},
    input="롤드컵 결승",
    tools=[
        {
            "type":"web_search", 
            "search_content_types":["image","text"],
            "image_settings":{
                "max_results":3,
                "caption":True
            }
        }
    ]
)

print(response.output_text)

2026 롤드컵 결승은 **11월 14일(토)**, 미국 뉴욕 브루클린의 **바클레이스 센터**에서 열려요. ([lolesports.com](https://lolesports.com/en-US/lolesports/news/msi-and-worlds-updates?utm_source=openai))

혹시 **결승 진출팀이나 경기 시간**을 물으신 거라면 말씀해 주세요.
